In [1]:
import torch
k = torch.load("/home/zhuominc/sglang/sgl-test/k.pt").cuda()
v = torch.load("/home/zhuominc/sglang/sgl-test/v.pt").cuda()
q = torch.load("/home/zhuominc/sglang/sgl-test/q.pt").cuda()
print(k.shape, v.shape, q.shape)

torch.Size([56, 4, 1, 128]) torch.Size([56, 4, 1, 128]) torch.Size([1, 2048])


In [2]:
# import torch
# ref_k = torch.load("/home/zhuominc/sglang/sgl-test/ref_k.pt").cuda()
# ref_v = torch.load("/home/zhuominc/sglang/sgl-test/ref_v.pt").cuda()
# ref_q = torch.load("/home/zhuominc/sglang/sgl-test/ref_q.pt").cuda()
# print(ref_k.shape, ref_v.shape, ref_q.shape)

In [3]:
# print(torch.allclose(q, ref_q))
# print(torch.allclose(k.view(224, 1, 128), ref_k))
# print(torch.allclose(v.view(224, 1, 128), ref_v))

In [4]:
from flashinfer import (
        BatchDecodeWithPagedKVCacheWrapper,
        BatchPrefillWithPagedKVCacheWrapper,
        BatchPrefillWithRaggedKVCacheWrapper,
    )

flashinfer_workspace_size = 512 * 1024 * 1024
workspace_buffer = torch.empty(
                flashinfer_workspace_size,
                dtype=torch.uint8,
                device="cuda",
            )

decode_wrappers = BatchDecodeWithPagedKVCacheWrapper(
                        workspace_buffer,
                        "NHD",
                        use_tensor_cores=True,
                )
        

2025-06-24 14:03:27,670 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend


In [5]:
# from torch import tensor

# ref_kv_indptr = tensor([  0,  16,  32], device='cuda:0',
#        dtype=torch.int32)
# ref_kv_indices = tensor([ 32,  33,  34,  35,  64,  65,  66,  67,  96,  97,  98,  99, 128, 129,
#         130, 131,  36,  37,  38,  39,  68,  69,  70,  71, 100, 101, 102, 103,
#         132, 133, 134, 135], device='cuda:0', dtype=torch.int32)
# ref_last_kv_len = tensor([1, 1], device='cuda:0', dtype=torch.int32)

# decode_wrappers.plan(
#             ref_kv_indptr,
#             ref_kv_indices,
#             ref_last_kv_len,
#             2,
#             1,
#             128,
#             1,
#             data_type=torch.bfloat16,
#             q_data_type=torch.bfloat16,
#             non_blocking=True,
#         )

# ref_q = ref_q.view(-1, 2, 128)[:2].contiguous()
# ref_o = decode_wrappers.run(
#             ref_q,
#             (ref_k,ref_v)
#         )

# print(ref_o[1])

In [ ]:
from torch import tensor

kv_indptr = tensor([ 0,  4,  8, 12, 16, 20, 24, 28, 32], device='cuda:0',
       dtype=torch.int32)
kv_indices = tensor([ 8, 16, 24, 32,  9, 17, 25, 33, 10, 18, 26, 34, 11, 19, 27, 35, 12, 20,
        28, 36, 13, 21, 29, 37, 14, 22, 30, 38, 15, 23, 31, 39],
       device='cuda:0', dtype=torch.int32)
last_kv_len = tensor([4, 4, 4, 4, 4, 4, 4, 4], dtype=torch.int32, device='cuda:0')

decode_wrappers.plan(
            kv_indptr,
            kv_indices,
            last_kv_len,
            2,
            1,
            128,
            4,
            data_type=torch.bfloat16,
            q_data_type=torch.bfloat16,
            non_blocking=False,
        )


o = decode_wrappers.run(
            q.contiguous().view(-1, 2, 128),
            (k,v)
        )

print(o)

tensor([[[-1.6602e-02, -4.9072e-02, -2.6978e-02,  ..., -3.3447e-02,
          -1.5625e-01,  8.7402e-02],
         [-2.8992e-03, -6.2256e-02, -4.3213e-02,  ..., -2.8687e-02,
          -1.8652e-01,  1.2988e-01]],

        [[-5.5420e-02, -7.2754e-02, -4.6143e-02,  ..., -7.1777e-02,
          -5.4443e-02,  1.0254e-02],
         [-1.8677e-02, -2.8809e-02, -1.1475e-01,  ..., -1.2268e-02,
          -4.7363e-02,  1.4551e-01]],

        [[ 1.7188e-01,  1.7944e-02, -5.3223e-02,  ...,  4.3213e-02,
           1.3672e+00,  6.1768e-02],
         [ 2.2266e-01,  1.9775e-02, -6.5430e-02,  ...,  5.4199e-02,
           1.9766e+00,  8.7891e-02]],

        ...,

        [[-1.0205e-01, -4.1016e-02, -2.8320e-02,  ...,  2.7222e-02,
           9.3384e-03,  5.1025e-02],
         [-4.7119e-02, -3.1738e-02, -1.5198e-02,  ...,  4.4678e-02,
          -3.9307e-02,  4.9316e-02]],

        [[-1.1963e-01,  9.5215e-02,  4.9316e-02,  ..., -9.8633e-02,
          -2.5391e-01,  9.2578e-01],
         [-9.7656e-03,  3.0029e-0

In [10]:
from torch import tensor

kv_indptr = tensor([ 0,  4,  8], device='cuda:0',
       dtype=torch.int32)
kv_indices = tensor([ 8, 16, 24, 32,  9, 17, 25, 33],
       device='cuda:0', dtype=torch.int32)
last_kv_len = tensor([4, 4], device='cuda:0')

decode_wrappers.plan(
            kv_indptr,
            kv_indices,
            last_kv_len,
            2,
            1,
            128,
            4,
            data_type=torch.bfloat16,
            q_data_type=torch.bfloat16,
            non_blocking=False,
        )


o = decode_wrappers.run(
            q.contiguous().view(-1, 2, 128)[:2],
            (k,v)
        )

print(o)

tensor([[[-1.6602e-02, -4.9072e-02, -2.6978e-02,  8.5449e-03,  1.0156e-01,
           4.8340e-02, -2.5635e-02,  2.8442e-02,  1.2305e-01,  2.4414e-02,
           1.0071e-02,  2.4170e-02,  9.7266e-01,  5.5237e-03, -2.3438e-02,
          -3.6865e-02, -1.6724e-02, -1.0889e-01, -1.4893e-02, -1.3379e-01,
          -1.3855e-02,  1.4355e-01, -3.3447e-02, -1.0107e-01, -3.1250e-02,
           6.1279e-02,  6.8848e-02,  8.2520e-02,  1.0156e-01, -1.8433e-02,
           1.2451e-02, -3.6377e-02,  2.5635e-02, -2.6953e-01, -1.5234e-01,
          -8.5938e-02, -7.9102e-02, -3.9307e-02, -6.8359e-02,  3.6621e-02,
           1.8848e-01,  1.2988e-01,  6.1035e-02,  1.2695e-02, -4.4434e-02,
          -7.6172e-02,  2.3804e-02,  3.3691e-02,  9.6191e-02, -1.1475e-02,
           4.1992e-02, -8.7891e-02, -1.6724e-02,  1.5736e-04,  6.1035e-02,
           8.0078e-02, -7.0312e-02,  5.7861e-02, -7.5000e-01, -9.9121e-02,
           5.3467e-02, -1.1536e-02, -3.1128e-02, -2.9419e-02, -4.2236e-02,
          -2.1744e-04, -1

In [7]:
#torch.allclose(o, ref_o)